# Degree Zero — Spanish pilot v2 (EN → ES, gloss baseline)

**Baseline G** is a deterministic word-for-word gloss of S (MUSE en–es dictionary, source word order) — a true "Degree Zero", not a model's attempt at being literal.

Conditions, all generated independently from S:

| | prompt | role |
|---|---|---|
| **L** | literal translation | does the LLM's "literal" sit closer to G than its free translation? |
| **F** | free translation | main condition |
| **F2** | free, reworded prompt | **noise floor**: how far apart do two equivalent prompts land? |
| **A** | free adaptation for teens | **positive control**: must sit far from G |

Main metric: **int_G(X) = ‖e(X) − e(G)‖**. The effect is real if int_G orders L < F < A *and* the gaps exceed the F↔F2 noise floor.

Adequacy here: LaBSE cross-lingual similarity (crude). The proper **QE metric (COMETKiwi)** runs in the companion notebook `degree_zero_qe.ipynb`.

**Runtime → Change runtime type → T4 GPU.**

In [ ]:
!pip install -q sentence-transformers sacrebleu accelerate
import torch, numpy as np, pandas as pd, re
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to GPU")

## Config

In [ ]:
GEN_MODEL   = "Qwen/Qwen3-4B-Instruct-2507"   # fallback if outputs look broken: "Qwen/Qwen2.5-3B-Instruct"
EMB_MODEL   = "intfloat/multilingual-e5-base"
XSIM_MODEL  = "sentence-transformers/LaBSE"   # cross-lingual S↔X similarity (crude adequacy)
N_SANITY    = 100
N_ITEMS     = 20      # per source type in "sot" mode
BATCH       = 8       # lower to 4 on CUDA OOM
MAX_NEW     = 512
RUN_SANITY     = False   # already passed; set True to re-run
RUN_BACKTRANS  = True    # round-trip drift (secondary to QE now)

SOURCE_MODE = "sot"
SOT_SOURCES = ["Human", "OpenAI"]
P_VERSION   = "chatgpt"
MIN_W, MAX_W = 80, 250
SEED = 0

## 1 · Load Ship of Theseus (+ optional sanity check)

In [ ]:
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer(EMB_MODEL, device="cuda")

def embed(texts):
    pre = "query: " if "e5" in EMB_MODEL else ""
    return emb_model.encode([pre + str(t) for t in texts], normalize_embeddings=True,
                            batch_size=64, show_progress_bar=False)

URL = "https://raw.githubusercontent.com/tripto03/Ship_of_theseus_paraphrased_copus/main/paraphrased_datasets/xsum_paraphrased.csv"
sot_all = pd.read_csv(URL)
sot = sot_all[sot_all.source == "Human"]
orig = sot[sot.version_name == "original"].drop_duplicates("key").set_index("key").text
keys = orig.sample(min(N_SANITY, len(orig)), random_state=SEED).index   # reserved for sanity check

if RUN_SANITY:
    rows = []
    for v in ["chatgpt", "chatgpt_chatgpt", "chatgpt_chatgpt_chatgpt",
              "pegasus(slight)", "pegasus(full)", "dipper(high)"]:
        sub = sot[sot.version_name == v].drop_duplicates("key").set_index("key").text
        k = keys.intersection(sub.index)
        d = np.linalg.norm(embed(sub[k]) - embed(orig[k]), axis=1)
        rows.append(dict(version=v, n=len(k), mean=d.mean(), median=np.median(d)))
    display(pd.DataFrame(rows).round(4))

## 2 · Source texts

In [ ]:
PASTED = """
Paste paragraph one here.

Paste paragraph two here.
"""

if SOURCE_MODE == "sot":
    o = sot_all[sot_all.version_name == "original"].drop_duplicates(["key", "source"])
    p = sot_all[sot_all.version_name == P_VERSION].drop_duplicates(["key", "source"])
    cand = o.merge(p[["key", "source", "text"]], on=["key", "source"], suffixes=("", "_P"))
    w = cand.text.str.split().str.len()
    cand = cand[w.between(MIN_W, MAX_W) & ~cand.key.isin(keys) & cand.source.isin(SOT_SOURCES)]
    df = (cand.sample(frac=1, random_state=SEED).groupby("source").head(N_ITEMS)
              .sort_values(["source", "key"])
              .rename(columns={"text": "S", "text_P": "P", "source": "source_type"})
              [["key", "source_type", "S", "P"]].reset_index(drop=True))
elif SOURCE_MODE == "paste":
    sources = [x.strip() for x in PASTED.split("\n\n") if len(x.split()) >= 30]
    df = pd.DataFrame({"S": sources[:N_ITEMS], "source_type": "pasted"})
else:
    from google.colab import files
    up = files.upload()
    df = pd.DataFrame({"S": pd.read_csv(next(iter(up))).text.dropna().tolist()[:N_ITEMS],
                       "source_type": "csv"})

print(df.source_type.value_counts().to_dict(), "| median words:", df.S.str.split().str.len().median())
assert len(df) >= 5, "Need more source paragraphs (each >= 30 words)"

## 3 · Gloss baseline G (deterministic, no model)
Word-for-word lookup in the MUSE en–es dictionary, source word order kept, punctuation untouched. Function words MUSE lacks get fixed glosses; unknown words (mostly names) are copied through. `gloss_cov` = share of words found.

In [ ]:
!wget -q -nc https://dl.fbaipublicfiles.com/arrival/dictionaries/en-es.txt

In [ ]:
# MUSE en-es lacks many function words; fixed glosses keep the baseline deterministic
FUNCTION_WORDS = {
    "a": "un", "an": "un", "the": "el", "of": "de", "to": "a", "in": "en", "into": "en", "on": "en",
    "at": "en", "for": "para", "by": "por", "with": "con", "from": "de", "about": "sobre", "as": "como",
    "and": "y", "or": "o", "but": "pero", "if": "si", "than": "que", "that": "que", "because": "porque",
    "is": "es", "are": "son", "was": "fue", "were": "fueron", "be": "ser", "been": "sido", "being": "siendo",
    "has": "ha", "have": "han", "had": "había", "will": "va", "would": "haría", "can": "puede",
    "could": "podría", "should": "debería", "may": "puede", "might": "podría", "must": "debe",
    "do": "hace", "does": "hace", "did": "hizo", "not": "no", "no": "no",
    "i": "yo", "you": "tú", "he": "él", "she": "ella", "it": "ello", "we": "nosotros", "they": "ellos",
    "me": "me", "him": "lo", "her": "su", "us": "nos", "them": "los",
    "my": "mi", "your": "tu", "his": "su", "its": "su", "our": "nuestro", "their": "su",
    "this": "este", "these": "estos", "those": "esos", "which": "cual", "who": "quien", "what": "qué",
    "up": "arriba", "out": "fuera", "over": "sobre", "after": "después", "before": "antes",
    "there": "allí", "here": "aquí", "also": "también", "very": "muy", "more": "más", "most": "más",
}
SUFFIXES = [("ies", "y"), ("ing", ""), ("ed", ""), ("es", ""), ("s", ""), ("ly", "")]
WORD = re.compile(r"([A-Za-z]+(?:'[A-Za-z]+)?)")


def load_muse(path):
    d = {}
    for line in open(path, encoding="utf-8"):
        parts = line.split()
        if len(parts) != 2:
            continue
        s, t = parts
        if s not in d or (d[s] == s and t != s):   # prefer first non-identity candidate
            d[s] = t
    d.update(FUNCTION_WORDS)
    return d


def gloss_word(w, d):
    lw = w.lower()
    if lw.endswith("'s"):                       # possessive: keep the noun, drop the clitic
        lw = lw[:-2]
    cands = [lw] + [lw[:-len(a)] + b for a, b in SUFFIXES if lw.endswith(a) and len(lw) - len(a) >= 3]
    for c in cands:
        if c in d:
            t = d[c]
            return (t.capitalize() if w[0].isupper() else t), True
    return w, False                             # OOV (names etc.): copy through


def gloss(text, d):
    """Word-for-word gloss in source order; punctuation and spacing kept as-is. Returns (gloss, coverage)."""
    out, hit, n = [], 0, 0
    for chunk in text.split():
        pieces = WORD.split(chunk)               # odd indices are words
        for i in range(1, len(pieces), 2):
            pieces[i], ok = gloss_word(pieces[i], d)
            n += 1
            hit += ok
        out.append("".join(pieces))
    return " ".join(out), (hit / n if n else 0.0)


muse = load_muse("en-es.txt")
df["G"], df["gloss_cov"] = zip(*[gloss(s, muse) for s in df.S])
print("gloss coverage median/min:", round(df.gloss_cov.median(), 3), round(df.gloss_cov.min(), 3))
print("S:", df.S[0][:250], "\nG:", df.G[0][:250])

## 4 · Generator + prompts

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(GEN_MODEL, padding_side="left")
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL, dtype=torch.float16, device_map="cuda")
gen_model.eval()

def chat(system, texts):
    outs = []
    for i in range(0, len(texts), BATCH):
        prompts = [tok.apply_chat_template([{"role": "system", "content": system},
                                            {"role": "user", "content": t}],
                                           tokenize=False, add_generation_prompt=True)
                   for t in texts[i:i + BATCH]]
        enc = tok(prompts, return_tensors="pt", padding=True).to(gen_model.device)
        with torch.no_grad():
            out = gen_model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                     temperature=None, top_p=None, top_k=None,
                                     pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        print(f"  {min(i + BATCH, len(texts))}/{len(texts)}")
    return [re.sub(r"<think>.*?</think>", "", o, flags=re.S).strip() for o in outs]

ONLY = " Output only the Spanish text, with no notes, quotes, or explanations."
PROMPTS = {
    "L":  "Translate the following English text into Spanish word-for-word. Preserve the source "
          "word order, sentence boundaries and structure as closely as Spanish grammar allows. "
          "Do not rephrase, merge, split, or reorder anything." + ONLY,
    "F":  "Translate the following English text into Spanish naturally and idiomatically, as a "
          "skilled professional translator would for a Spanish-speaking readership." + ONLY,
    "F2": "Render this English news text in fluent, natural Spanish, the way an experienced "
          "newsroom translator would." + ONLY,
    "A":  "Freely adapt the following English text into Spanish for a teenage audience: simplify "
          "vocabulary, restructure sentences and rephrase as much as you like, but keep the key facts." + ONLY,
    "back": "Translate the following Spanish text into English. Output only the translation, with no notes.",
}

## 5 · Generate

In [ ]:
for c in ["L", "F", "F2", "A"]:
    print(c); df[c] = chat(PROMPTS[c], df.S.tolist())
if RUN_BACKTRANS:
    print("B_L"); df["B_L"] = chat(PROMPTS["back"], df.L.tolist())
    print("B_F"); df["B_F"] = chat(PROMPTS["back"], df.F.tolist())
df.to_csv("/content/generations.csv", index=False)

for _, r in df.head(2).iterrows():
    for c in ["S", "G", "L", "F", "F2", "A"]:
        print(f"{c:2s}:", r[c][:220])
    print("-" * 80)

## 6 · Flag broken generations
Last pilot: Qwen2.5-3B drifted into Chinese or left English untranslated in 4/40 literal outputs, and those rows dominated every statistic. Flagged rows are kept in the CSV (`bad=True`) but excluded from the analysis.

In [ ]:
COND = ["L", "F", "F2", "A"]
has_cjk = lambda s: bool(re.search(r"[\u3040-\u30ff\u4e00-\u9fff\uac00-\ud7af]", str(s)))
en_rate = lambda s: len(re.findall(r"\b(the|and|of|was|is|that|with)\b", str(s), re.I)) / max(1, len(str(s).split()))
df["bad_reason"] = ""
for c in COND:
    df.loc[df[c].map(has_cjk), "bad_reason"] += f"{c}:cjk "
    df.loc[df[c].map(en_rate) > 0.05, "bad_reason"] += f"{c}:english "
df["bad"] = df.bad_reason != ""
print(f"{df.bad.sum()} / {len(df)} rows flagged"); display(df.loc[df.bad, ["key", "source_type", "bad_reason"]])

## 7 · Metrics

In [ ]:
import sacrebleu
xsim_model = SentenceTransformer(XSIM_MODEL, device="cuda")
xembed = lambda t: xsim_model.encode(list(map(str, t)), normalize_embeddings=True, batch_size=64)

E = {c: embed(df[c]) for c in ["S", "G"] + COND}
XS = xembed(df.S)
chrf = lambda hyp, ref: [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(hyp, ref)]

for c in COND:
    df[f"int_G_{c}"] = np.linalg.norm(E[c] - E["G"], axis=1)   # distance from Degree Zero
    df[f"chrF_G_{c}"] = chrf(df[c], df.G)                       # surface overlap with gloss
    df[f"xsim_{c}"] = (XS * xembed(df[c])).sum(1)               # crude adequacy vs S
df["xsim_G"] = (XS * xembed(df.G)).sum(1)
df["floor_F"] = np.linalg.norm(E["F"] - E["F2"], axis=1)        # rewording noise
df["d_LF"] = np.linalg.norm(E["L"] - E["F"], axis=1)
df["chrF_F_F2"] = chrf(df.F2, df.F)

if RUN_BACKTRANS:
    eS = E["S"]
    df["drift_L"] = 1 - (eS * embed(df.B_L)).sum(1)
    df["drift_F"] = 1 - (eS * embed(df.B_F)).sum(1)
    df["excess_drift"] = df.drift_F - df.drift_L
if "P" in df:
    df["paraphrase_vec"] = np.linalg.norm(embed(df.P) - E["S"], axis=1)

df.to_csv("/content/results.csv", index=False)
ok = df[~df.bad]
print("clean n =", len(ok))
display(pd.DataFrame({c: {"int_G": ok[f"int_G_{c}"].median(), "chrF vs G": ok[f"chrF_G_{c}"].median(),
                          "xsim vs S": ok[f"xsim_{c}"].median()} for c in COND}).T.round(3))
print("noise floor ‖F−F2‖ median:", round(ok.floor_F.median(), 4), "| ‖L−F‖ median:", round(ok.d_LF.median(), 4))

## 8 · Is there an effect?
1. **Positive control**: int_G(A) > int_G(F). If this fails, the embedding can't see restructuring in Spanish — stop.
2. **Literal vs free**: int_G(L) < int_G(F), and ‖L−F‖ > noise floor ‖F−F2‖. If ‖L−F‖ ≈ floor, the literal prompt is not producing anything different.
3. **Validation**: int_G should correlate negatively with chrF vs G, but not near −1 (else it's just surface overlap).

In [ ]:
from scipy.stats import wilcoxon, spearmanr, mannwhitneyu
def paired(a, b, label):
    diff = ok[a] - ok[b]
    p = wilcoxon(ok[a], ok[b], alternative="greater").pvalue
    print(f"{label:38s} median diff={diff.median():+.4f}  {(diff > 0).mean():.0%} of items  p={p:.3g}")
    return p

p_ctrl = paired("int_G_A", "int_G_F", "positive control  int_G(A) > int_G(F)")
p_lf   = paired("int_G_F", "int_G_L", "literal vs free   int_G(F) > int_G(L)")
p_fl   = paired("d_LF", "floor_F", "signal vs noise   ‖L−F‖ > ‖F−F2‖")
print()
print("✅ embedding sees restructuring" if p_ctrl < 0.05 else "❌ positive control failed — embedding insensitive in Spanish")
print("✅ literal prompt differs beyond rewording noise" if p_fl < 0.05 else "⚠️ ‖L−F‖ not above noise floor — LLM 'literal' ≈ free")

pooled = pd.concat([pd.DataFrame({"int_G": ok[f"int_G_{c}"], "chrF": ok[f"chrF_G_{c}"]}) for c in COND])
print("\nvalidation: spearman int_G vs chrF-vs-G (pooled) =", round(spearmanr(pooled.int_G, pooled.chrF).statistic, 3))
if ok.source_type.nunique() == 2:
    a, b = [g for _, g in ok.groupby("source_type")]
    print("Human vs OpenAI int_G(F):", a.int_G_F.median().round(4), b.int_G_F.median().round(4),
          " p =", round(mannwhitneyu(a.int_G_F, b.int_G_F).pvalue, 3))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6.5, 4.5))
x = np.arange(len(COND))
for _, r in ok.iterrows():
    ax.plot(x, [r[f"int_G_{c}"] for c in COND], c="gray", alpha=0.25, lw=1)
ax.plot(x, [ok[f"int_G_{c}"].median() for c in COND], c="C0", lw=3, marker="o", label="median")
ax.axhspan(0, ok.floor_F.median(), color="C1", alpha=0.12, label="median noise floor ‖F−F2‖")
ax.set_xticks(x, ["L literal", "F free", "F2 free (reworded)", "A adapt"])
ax.set_ylabel("int_G  =  ‖e(X) − e(G)‖"); ax.set_title(f"{GEN_MODEL.split('/')[-1]} · distance from gloss baseline")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
from google.colab import files
files.download("/content/results.csv")